In [1]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps

In [2]:
from datasets import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

import evaluate
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer,
                          EarlyStoppingCallback)

# 1. Data Loading

In [35]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


In [36]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
train_data['full_text'] = train_data.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
kaggle_data['full_text'] = kaggle_data.apply(lambda tweet: extract_full_text(tweet), axis=1)

In [44]:

# =======================
# 0) Configs / utilitaires
# =======================
SEED = 10
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "vinai/bertweet-base"   # ou "roberta-base"
TEXT_COL   = "full_text"
LABEL_COL  = "label"
ID_COL     = "id_str"              # pour la soumission
MAX_LEN    = 100

def normalize_tweet(text: str) -> str:
    """Petit nettoyage non destructif: conserve le signal (mentions, hashtags, emojis)."""
    if not isinstance(text, str):
        return ""
    # simplifier les URLs -> 'http'
    t = text.replace("https://", "http ").replace("http://", "http ")
    return t


def preprocess(batch):
    texts = batch["full_text"]  # adapte le nom de ta colonne texte
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length"  # on laisse le collator gérer le padding dynamique
    )

# 1) Construire des jeux "propres" (enlever TOUT le reste)
cols_keep = ["full_text", "label"]  # pas d’autres colonnes !
ds_tr = Dataset.from_pandas(df_tr[cols_keep].reset_index(drop=True))
ds_va = Dataset.from_pandas(df_va[cols_keep].reset_index(drop=True))

# 2) Tokeniser
ds_tr = ds_tr.map(preprocess, batched=True, remove_columns=ds_tr.column_names)  # <- retire les anciennes colonnes
ds_va = ds_va.map(preprocess, batched=True, remove_columns=ds_va.column_names)

# 3) (ré)ajouter les labels après map si remove_columns les a enlevés
ds_tr = ds_tr.add_column("labels", df_tr["label"].astype(int).tolist())
ds_va = ds_va.add_column("labels", df_va["label"].astype(int).tolist())

# 4) Formater pour torch avec UNIQUEMENT ces colonnes
model_input_cols = ["input_ids", "attention_mask", "labels"]
ds_tr = ds_tr.with_format("torch", columns=model_input_cols)
ds_va = ds_va.with_format("torch", columns=model_input_cols)

# 5) Collator qui padde de façon cohérente toutes les clés d’inputs
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

Map:   0%|          | 0/310 [00:00<?, ? examples/s]

In [45]:
#### on teste sur petit volume de données : 

FRAC = 0.01          # 1%
LABEL_COL = "label"  # adapte si besoin

def stratified_fraction(df, label_col=LABEL_COL, frac=FRAC, min_per_class=1, seed=42):
    out = []
    for y, g in df.groupby(label_col):
        n = max(min_per_class, int(round(len(g) * frac)))
        n = min(n, len(g))  # sécurité
        out.append(g.sample(n=n, random_state=seed))
    return pd.concat(out).sample(frac=1.0, random_state=seed).reset_index(drop=True)

# Exemple d'usage sur ton train
df_small = stratified_fraction(train_data, LABEL_COL, FRAC)
print(df_small.shape, df_small[LABEL_COL].value_counts(normalize=True))

# (Optionnel) réduire aussi le test Kaggle (juste pour la vitesse de test du pipeline)
X_kaggle_small = X_kaggle.sample(n=min(2000, len(X_kaggle)), random_state=42)

(1549, 194) label
0    0.533893
1    0.466107
Name: proportion, dtype: float64


In [46]:
# =======================
# 1) Charger les données
# =======================
# Remplace ces deux lignes par tes DataFrames déjà en mémoire si besoin.
# df_train = pd.read_csv("train.csv")     # doit contenir TEXT_COL et LABEL_COL
# X_kaggle = pd.read_csv("test.csv")      # doit contenir TEXT_COL et ID_COL

# Nettoyage minimal / sécurité
df = df_small[[TEXT_COL, LABEL_COL]].dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)
X_kaggle = kaggle_data[[TEXT_COL, ID_COL]].copy()


In [47]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, va_idx = next(sss.split(df, df[LABEL_COL]))
df_tr, df_va = df.iloc[tr_idx].reset_index(drop=True), df.iloc[va_idx].reset_index(drop=True)

print("Split:", df_tr.shape, df_va.shape, " | Ratio labels (train):",
      df_tr[LABEL_COL].mean(), " | (val):", df_va[LABEL_COL].mean())

Split: (1239, 2) (310, 2)  | Ratio labels (train): 0.4665052461662631  | (val): 0.4645161290322581


In [48]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess(batch):
    texts = [normalize_tweet(t) for t in batch[TEXT_COL]]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length"
    )

ds_tr = Dataset.from_pandas(df_tr)
ds_va = Dataset.from_pandas(df_va)
ds_tr = ds_tr.map(preprocess, batched=True)
ds_va = ds_va.map(preprocess, batched=True)

cols_model = ["input_ids", "attention_mask"]
ds_tr = ds_tr.rename_columns({LABEL_COL: "labels"}).with_format("torch", columns=cols_model+["labels"])
ds_va = ds_va.rename_columns({LABEL_COL: "labels"}).with_format("torch", columns=cols_model+["labels"])


Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

Map:   0%|          | 0/310 [00:00<?, ? examples/s]

In [49]:
# =======================
# 4) Modèle + class weights
# =======================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Poids de classes (optionnel, utile si léger déséquilibre)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=df_tr[LABEL_COL].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

def compute_weighted_loss(model, inputs, return_outputs=False,**kwargs):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits
    loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
    loss = loss_fct(logits, labels)
    return (loss, outputs) if return_outputs else loss

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [50]:
# =======================
# 5) Entraînement
# =======================
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./checkpoints_ft",
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=torch.cuda.is_available(),   # True si GPU BF16; sinon False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    eval_dataset=ds_va,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
# injecter la loss pondérée (sinon commente la ligne ci-dessous pour loss standard)
trainer.compute_loss = compute_weighted_loss

trainer.train()
print(trainer.evaluate())

C:\Users\poule\AppData\Local\Temp\ipykernel_4664\2981280685.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\poule\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1
200,0.632700,0.671601,0.593548,0.593548


C:\Users\poule\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\poule\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.6716011762619019, 'eval_accuracy': 0.5935483870967742, 'eval_f1': 0.5935483870967742, 'eval_runtime': 26.7716, 'eval_samples_per_second': 11.579, 'eval_steps_per_second': 0.374, 'epoch': 4.0}


# 3. Logistic Regression Classifier

In [13]:
import transformers
print(transformers.__version__)

4.57.1


In [4]:
# Load a list of common French stop words (e.g., 'le', 'la', 'de')
french_stop_words = stopwords.words('french')

print("\nBuilding model pipeline...")

# Create a scikit-learn Pipeline. This chains steps together.
# Data will flow from 'tfidf' (text to numbers) to 'clf' (classifier).
model_pipeline = Pipeline([
    # Step 1: TfidfVectorizer - converts text into a matrix of TF-IDF features
    ('tfidf', TfidfVectorizer(
        stop_words=french_stop_words, # Remove French stop words
        max_df=0.7,       # Ignore words that appear in > 70% of tweets (too common)
        min_df=3,         # Ignore words that appear in < 3 tweets (too rare)
        max_features=1000, # Keep only the top 1000 features
        ngram_range=(1, 2)  # Include 1-word (unigrams) and 2-word (bigrams) sequences
    )),
    # Step 2: Classifier - Logistic Regression
    ('clf', LogisticRegression(
        random_state=42,    # For reproducible results
        solver='liblinear'  # Good solver for this type of problem
    ))
])

print("\nRunning 5-Fold Cross-Validation on training data...")

# Use StratifiedKFold to ensure class proportions are maintained in each fold
# This is important for datasets that might be imbalanced
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_score will train and test the pipeline 5 times
# using the K-fold splits of the *training data*
scores = cross_val_score(
    model_pipeline,          # The pipeline to evaluate
    X_train['full_text'],  # Features from training set
    y_train,               # Labels from training set
    cv=kfold,              # The stratified 5-fold splitter
    scoring='accuracy'     # The metric to evaluate
)

# Print the cross-validation results
print(f"K-Fold Accuracy Scores: {scores}")
print(f"Mean K-Fold Accuracy: {np.mean(scores) * 100:.2f}%")
print(f"Std Dev K-Fold Accuracy: {np.std(scores) * 100:.2f}%")


print("\nTraining final model on all training data...")
# Now that we've validated the model, train it on ALL available training data
model_pipeline.fit(X_train['full_text'], y_train)
print("Training complete.")

print("\n--- Final Model Evaluation on Held-Out Test Set ---")
# Use the trained pipeline to make predictions on the unseen Kaggle data
# The pipeline automatically applies the TF-IDF transform and then predicts
y_pred_test = model_pipeline.predict(X_kaggle['full_text'])

# Prepare the submission file
# Combine the 'challenge_id' from the Kaggle data with our predictions
output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_test)], axis=1,ignore_index=True)
# Rename columns to match the required submission format
output.columns = ['ID', "Prediction"]
# Save the submission file as a CSV
output.to_csv('logistic_regression.csv', index=False)


Building model pipeline...

Running 5-Fold Cross-Validation on training data...
K-Fold Accuracy Scores: [0.62589162 0.6281832  0.62066294 0.62337411 0.63036602]
Mean K-Fold Accuracy: 62.57%
Std Dev K-Fold Accuracy: 0.34%

Training final model on all training data...
Training complete.

--- Final Model Evaluation on Held-Out Test Set ---


# 4. Dummy Classifier

In [5]:
print("\nTraining Dummy (Most Frequent)...")
# Create a DummyClassifier that always predicts the most frequent class
# This is a baseline to see if our Logistic Regression model is actually learning anything
dummy_mf = DummyClassifier(strategy="most_frequent")

# "Train" the dummy model (it just finds the most frequent class in y_train)
dummy_mf.fit(X_train['full_text'], y_train)

# Make predictions on the Kaggle data (it will predict the same class for all rows)
y_pred_test = dummy_mf.predict(X_kaggle['full_text'])

# Prepare and save the dummy submission file
output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_test)], axis=1,ignore_index=True)
output.columns = ['ID', "Prediction"]
output.to_csv('dummy.csv', index=False)


Training Dummy (Most Frequent)...
